# Above 8000

Fourteen mountains on Earth rise above 8,000 meters — the so-called eight-thousanders. All of them stand within the Himalayan and Karakoram ranges, clustered along the collision zone where the Indian subcontinent still presses northward into Asia. Their summits reach into the jet stream, where hurricane-force winds and temperatures below −40°C test the limits of human endurance.

Between 1950 and 1964, in what alpinists call the Golden Age of Himalayan mountaineering, expedition teams from a dozen nations reached the summit of every one of these peaks for the first time. The era began with Annapurna in 1950 — a French expedition led by Maurice Herzog — and closed fourteen years later when a Chinese team stood atop Shishapangma. In just over a decade, humanity had touched every summit above 8,000 meters.

In the decades since, these peaks have drawn thousands of climbers seeking their own moment above the clouds. Everest alone has seen nearly 13,000 successful ascents, while Annapurna — the first eight-thousander ever climbed — remains one of the most exclusive, with fewer than 400 people having reached its summit. Each mountain has its own character: some are relatively accessible, others remain formidable challenges that only a handful attempt each year.

This notebook visualizes all fourteen eight-thousanders as a procedural mountain panorama. Each peak's silhouette is generated from Perlin noise, arranged by height and animated in the order they were first climbed. After the range takes shape, a color transition highlights how many climbers have stood on each summit — from the rarest ascents to the most popular.

In [ ]:
import chroma from 'chroma-js';

## The Data

Statistics are compiled from the [Himalayan Database](https://www.himalayandatabase.com/) and multiple mountaineering sources as of 2024. Each peak carries its height, first ascent date and team, and the total number of successful ascents recorded across decades of expeditions. Heights follow the most recent official surveys.

In [ ]:
// 14 eight-thousanders: height, first ascent, expedition stats
// sources: Himalayan Database, himaldb.com, as of 2024
const peaks = [
  { name: 'Everest',       height: 8849, range: 'Mahalangur Himalaya', country: 'Nepal / China',    firstAscent: 1953, climbers: 'Hillary & Norgay',                summits: 12884, deaths: 326, seed: 1  },
  { name: 'K2',            height: 8611, range: 'Baltoro Karakoram',   country: 'Pakistan / China', firstAscent: 1954, climbers: 'Compagnoni & Lacedelli',          summits: 800,   deaths: 96,  seed: 2  },
  { name: 'Kangchenjunga', height: 8586, range: 'Kangchenjunga Himal', country: 'Nepal / India',    firstAscent: 1955, climbers: 'Brown & Band',                    summits: 532,   deaths: 52,  seed: 3  },
  { name: 'Lhotse',        height: 8516, range: 'Mahalangur Himalaya', country: 'Nepal / China',    firstAscent: 1956, climbers: 'Luchsinger & Reiss',              summits: 1089,  deaths: 22,  seed: 4  },
  { name: 'Makalu',        height: 8485, range: 'Mahalangur Himalaya', country: 'Nepal / China',    firstAscent: 1955, climbers: 'Couzy & Terray',                  summits: 647,   deaths: 48,  seed: 5  },
  { name: 'Cho Oyu',       height: 8188, range: 'Mahalangur Himalaya', country: 'Nepal / China',    firstAscent: 1954, climbers: 'Tichy, J\u00f6chler & Pasang Dawa',    summits: 4027,  deaths: 52,  seed: 6  },
  { name: 'Dhaulagiri I',  height: 8167, range: 'Dhaulagiri Himalaya', country: 'Nepal',            firstAscent: 1960, climbers: 'Diemberger, Schelbert & team',     summits: 530,   deaths: 83,  seed: 7  },
  { name: 'Manaslu',       height: 8163, range: 'Manaslu Himalaya',    country: 'Nepal',            firstAscent: 1956, climbers: 'Imanishi & Gyalzen Norbu',         summits: 1817,  deaths: 90,  seed: 8  },
  { name: 'Nanga Parbat',  height: 8126, range: 'Nanga Parbat Himal',  country: 'Pakistan',         firstAscent: 1953, climbers: 'Buhl (solo)',                      summits: 440,   deaths: 85,  seed: 9  },
  { name: 'Annapurna I',   height: 8091, range: 'Annapurna Himalaya',  country: 'Nepal',            firstAscent: 1950, climbers: 'Herzog & Lachenal',                summits: 365,   deaths: 73,  seed: 10 },
  { name: 'Gasherbrum I',  height: 8080, range: 'Baltoro Karakoram',   country: 'Pakistan / China', firstAscent: 1958, climbers: 'Schoening & Kauffman',             summits: 393,   deaths: 32,  seed: 11 },
  { name: 'Broad Peak',    height: 8051, range: 'Baltoro Karakoram',   country: 'Pakistan / China', firstAscent: 1957, climbers: 'Schmuck, Wintersteller & team',    summits: 510,   deaths: 24,  seed: 12 },
  { name: 'Gasherbrum II', height: 8035, range: 'Baltoro Karakoram',   country: 'Pakistan / China', firstAscent: 1956, climbers: 'Moravec, Larch & Willenpart',      summits: 1200,  deaths: 23,  seed: 13 },
  { name: 'Shishapangma',  height: 8027, range: 'Jugal Himalaya',      country: 'China',            firstAscent: 1964, climbers: 'Xu Jing & Chinese team',           summits: 340,   deaths: 28,  seed: 14 },
];

const minHeight = Math.min(...peaks.map(p => p.height));
const maxHeight = Math.max(...peaks.map(p => p.height));

// Sort by height for display (tallest left)
const byHeight = [...peaks].sort((a, b) => b.height - a.height);

// Sort by first ascent for animation order
const byAscent = [...peaks].sort((a, b) => a.firstAscent - b.firstAscent || a.height - b.height);

// Render as HTML table
const css = 'border-collapse:collapse;font-family:monospace;font-size:13px;width:100%';
const thStyle = 'padding:6px 10px;text-align:left;border-bottom:2px solid #444;color:#8aa';
const tdStyle = 'padding:5px 10px;border-bottom:1px solid #333';
const tdR = 'padding:5px 10px;border-bottom:1px solid #333;text-align:right';

const header = '<tr>' +
  ['Peak', 'Height (m)', 'Range', 'Country', 'First Ascent', 'Climbers', 'Summits']
    .map(h => '<th style="' + thStyle + '">' + h + '</th>').join('') +
  '</tr>';

const rows = byHeight.map(p =>
  '<tr>' +
  '<td style="' + tdStyle + ';font-weight:bold">' + p.name + '</td>' +
  '<td style="' + tdR + '">' + p.height.toLocaleString() + '</td>' +
  '<td style="' + tdStyle + '">' + p.range + '</td>' +
  '<td style="' + tdStyle + '">' + p.country + '</td>' +
  '<td style="' + tdR + '">' + p.firstAscent + '</td>' +
  '<td style="' + tdStyle + '">' + p.climbers + '</td>' +
  '<td style="' + tdR + '">' + p.summits.toLocaleString() + '</td>' +
  '</tr>'
).join('');

const totalSummits = peaks.reduce((s, p) => s + p.summits, 0);

'<table style="' + css + '">' +
  '<thead>' + header + '</thead>' +
  '<tbody>' + rows + '</tbody>' +
  '<tfoot><tr>' +
    '<td style="' + tdStyle + ';font-weight:bold" colspan="6">Total (' + peaks.length + ' peaks, ' + minHeight + '\u2013' + maxHeight + ' m)</td>' +
    '<td style="' + tdR + ';font-weight:bold">' + totalSummits.toLocaleString() + '</td>' +
  '</tr></tfoot>' +
'</table>'

## The Summit Circle

How many people have stood on top of each peak? The answer varies enormously. Everest, with its established routes and commercial expeditions, has welcomed nearly 13,000 summiteers. Cho Oyu, considered the most accessible eight-thousander, has seen over 4,000. At the other end, Annapurna and Shishapangma remain among the most exclusive — fewer than 400 summits each. The color transition maps this spectrum: warm amber for the rarest achievements, cool blue for the well-traveled summits.

In [ ]:
// warm amber (rare) → cool blue (popular)
const summitScale = chroma
  .scale(['#e76f51', '#f4a261', '#e9c46a', '#a8dadc', '#457b9d', '#1d3557'])
  .domain([300, 500, 1000, 2000, 5000, 13000])
  .mode('lab');

const params = {
  bgColor: [5, 6, 18],
  labelColor: [170, 190, 215],
  framesPerPeak: 40,
  holdFrames: 80,
  transitionFrames: 120,
  profileSteps: 140,
  mountainGap: 4,
};

// summit count color preview
byHeight.map(p => ({
  peak: p.name,
  summits: p.summits.toLocaleString(),
  color: summitScale(p.summits).hex(),
}))

## Mountain Profiles

Each peak’s silhouette is generated from Perlin noise, producing a unique ridge line for every mountain. The profiles are shaped by an asymmetric envelope — a rounded triangle whose summit is offset slightly from center — layered with medium-scale ridge features and fine-scale rock detail. The result is fourteen distinct silhouettes that feel geologically plausible, each seeded by the peak’s index for reproducibility.

In [ ]:
function fbm(x, seed, octaves) {
  let val = 0;
  let amp = 1;
  let freq = 1;
  let total = 0;
  for (let o = 0; o < octaves; o++) {
    val += noise(x * freq + seed + o * 17.31) * amp;
    total += amp;
    amp *= 0.5;
    freq *= 2.2;
  }
  return val / total;
}

function generateProfile(seed) {
  let points = [];
  let steps = params.profileSteps;

  // unique shape characteristics per peak
  let peakPos = 0.38 + fbm(seed * 0.1, seed * 73.7, 3) * 0.24;
  let steepness = 2.8 + fbm(seed * 0.2, seed * 41.1, 2) * 2.0;
  let hasShoulderLeft = noise(seed * 31.1) > 0.55;
  let hasShoulderRight = noise(seed * 19.7) > 0.55;
  let shoulderHeight = 0.35 + noise(seed * 41.3) * 0.3;
  let jaggedness = 0.25 + noise(seed * 61.7) * 0.2;

  for (let i = 0; i <= steps; i++) {
    let t = i / steps;

    // gaussian main envelope (no triangle)
    let mainEnv = exp(-pow((t - peakPos) * steepness, 2));

    // optional shoulder sub-peaks
    let shoulder = 0;
    if (hasShoulderLeft) {
      let sPos = peakPos * 0.35;
      shoulder = max(shoulder, exp(-pow((t - sPos) * 5.5, 2)) * shoulderHeight);
    }
    if (hasShoulderRight) {
      let sPos = peakPos + (1 - peakPos) * 0.6;
      shoulder = max(shoulder, exp(-pow((t - sPos) * 5.0, 2)) * (shoulderHeight * 0.8));
    }

    // blended envelope
    let env = max(mainEnv, shoulder * 0.7) + shoulder * 0.15;
    env = min(env, 1.15);

    // 6 octaves of fractal ridge noise
    let ridgeNoise = (fbm(t * 6, seed * 13.7, 6) - 0.5) * jaggedness;

    let h = env + ridgeNoise * env;

    // taper edges to zero
    let edgeFade = pow(sin(constrain(t, 0.001, 0.999) * PI), 0.35);
    h *= edgeFade;

    points.push(max(h, 0));
  }

  let mx = Math.max(...points);
  return points.map(h => h / mx);
}

## The Panorama

The sketch arranges all fourteen peaks by height — Everest on the left, Shishapangma on the right — forming a descending ridge line. Each mountain rises in the order it was first climbed, from Annapurna (1950) to Shishapangma (1964). Once the range is complete, the colors shift from natural rock and snow tones to the summit scale, revealing which peaks are the most and least frequented. Hover over any mountain for its full story.

In [ ]:
let currentIndex = 0;
let playing = true;
let timeline, speedSlider, playBtn;
let baseY, maxDisplayH;
let mountainData = [];
let riseEnd, holdEnd, totalFrames;

function easeOutCubic(t) {
  return 1 - pow(1 - t, 3);
}

function easeInOutCubic(t) {
  return t < 0.5 ? 4 * t * t * t : 1 - pow(-2 * t + 2, 3) / 2;
}

function buildPath(ctx, profile, x, w, h) {
  ctx.beginPath();
  for (let i = 0; i <= profile.length; i++) {
    let t = i / profile.length;
    let px = x - w / 2 + t * w;
    let idx = min(i, profile.length - 1);
    let py = baseY - profile[idx] * h;
    if (i === 0) ctx.moveTo(px, py);
    else ctx.lineTo(px, py);
  }
}

function drawMountain(md, progress, colorBlend) {
  let x = md.displayX;
  let w = md.displayW;
  let h = md.displayH * easeOutCubic(progress);
  let profile = md.profile;

  let ctx = drawingContext;
  ctx.save();

  let grad = ctx.createLinearGradient(x, baseY - h, x, baseY);

  if (colorBlend > 0.01) {
    let sc = summitScale(md.summits);
    let top = chroma.mix('#eef2f6', sc.brighten(2.0).hex(), colorBlend).hex();
    let snow = chroma.mix('#c2d0dc', sc.brighten(1.2).hex(), colorBlend).hex();
    let rock1 = chroma.mix('#6a7a88', sc.brighten(0.2).hex(), colorBlend).hex();
    let rock2 = chroma.mix('#38484e', sc.hex(), colorBlend).hex();
    let base = chroma.mix('#182028', sc.darken(1.8).hex(), colorBlend).hex();
    grad.addColorStop(0, top);
    grad.addColorStop(0.12, snow);
    grad.addColorStop(0.35, rock1);
    grad.addColorStop(0.65, rock2);
    grad.addColorStop(1, base);
  } else {
    grad.addColorStop(0, '#eef2f6');
    grad.addColorStop(0.12, '#bccad8');
    grad.addColorStop(0.28, '#6e7e8e');
    grad.addColorStop(0.55, '#384a56');
    grad.addColorStop(0.8, '#202c36');
    grad.addColorStop(1, '#121c24');
  }

  ctx.fillStyle = grad;

  buildPath(ctx, profile, x, w, h);
  ctx.lineTo(x + w / 2, baseY);
  ctx.lineTo(x - w / 2, baseY);
  ctx.closePath();
  ctx.fill();

  // ridge highlight
  buildPath(ctx, profile, x, w, h);
  let ridgeAlpha = colorBlend > 0.01 ? 0.12 + colorBlend * 0.08 : 0.22;
  let ridgeColor = colorBlend > 0.01
    ? summitScale(md.summits).brighten(2.5).alpha(ridgeAlpha).css()
    : 'rgba(170, 195, 225, ' + ridgeAlpha + ')';
  ctx.strokeStyle = ridgeColor;
  ctx.lineWidth = 0.8;
  ctx.stroke();

  // subtle inner glow along ridgeline
  buildPath(ctx, profile, x, w, h);
  ctx.strokeStyle = 'rgba(130, 160, 200, 0.06)';
  ctx.lineWidth = 3;
  ctx.stroke();

  // summit glow during rise
  if (progress > 0.4 && progress < 1) {
    let glowAlpha = sin((progress - 0.4) / 0.6 * PI) * 0.45;
    let peakIdx = profile.indexOf(Math.max(...profile));
    let peakX = x - w / 2 + (peakIdx / (profile.length - 1)) * w;
    let peakY = baseY - h;

    let rg = ctx.createRadialGradient(peakX, peakY, 0, peakX, peakY, 35);
    rg.addColorStop(0, 'rgba(255, 255, 255, ' + glowAlpha + ')');
    rg.addColorStop(0.4, 'rgba(200, 220, 255, ' + glowAlpha * 0.3 + ')');
    rg.addColorStop(1, 'rgba(200, 220, 255, 0)');
    ctx.fillStyle = rg;
    ctx.beginPath();
    ctx.arc(peakX, peakY, 35, 0, Math.PI * 2);
    ctx.fill();
  }

  ctx.restore();
}

function getHoveredPeak() {
  for (let md of mountainData) {
    let riseProgress = constrain(
      (currentIndex - md.ascentIdx * params.framesPerPeak) / params.framesPerPeak, 0, 1
    );
    if (riseProgress < 1) continue;

    let halfW = md.displayW / 2;
    if (mouseX >= md.displayX - halfW && mouseX <= md.displayX + halfW) {
      let t = (mouseX - (md.displayX - halfW)) / md.displayW;
      let idx = constrain(floor(t * md.profile.length), 0, md.profile.length - 1);
      let peakY = baseY - md.profile[idx] * md.displayH;
      if (mouseY >= peakY && mouseY <= baseY) {
        return md;
      }
    }
  }
  return null;
}

function drawTooltip(md) {
  let tx = md.displayX;
  let ty = baseY - md.displayH - 16;

  let boxW = 210;
  let boxH = 110;
  tx = constrain(tx, boxW / 2 + 10, width - boxW / 2 - 10);
  ty = max(ty, boxH + 20);

  let ctx = drawingContext;
  ctx.save();
  ctx.shadowColor = 'rgba(0, 0, 0, 0.5)';
  ctx.shadowBlur = 16;
  ctx.shadowOffsetY = 4;
  fill(8, 10, 22, 235);
  stroke(100, 120, 150, 35);
  strokeWeight(1);
  rect(tx - boxW / 2, ty - boxH, boxW, boxH, 8);
  ctx.restore();

  noStroke();
  textAlign(LEFT, TOP);
  let lx = tx - boxW / 2 + 14;
  let ly = ty - boxH + 12;

  fill(255, 245);
  textSize(14);
  text(md.name, lx, ly);

  fill(140, 160, 190, 180);
  textSize(10);
  text(md.height + ' m  \u00b7  ' + md.range, lx, ly + 20);
  text(md.country, lx, ly + 34);

  fill(180, 190, 210, 200);
  textSize(11);
  text('First ascent ' + md.firstAscent, lx, ly + 54);
  textSize(9);
  fill(140, 160, 190, 150);
  text(md.climbers, lx, ly + 68);

  let sc = summitScale(md.summits).rgb();
  fill(sc[0], sc[1], sc[2], 240);
  textSize(11);
  text('Summits: ' + md.summits.toLocaleString(), lx, ly + 88);

  textAlign(CENTER, CENTER);
}

## Putting It Together

The sketch ties everything into a single animated canvas. It plays in two acts: first the panorama builds peak by peak in order of first ascent, then a color transition maps each mountain's summit count onto the rarity scale — warm amber for the most exclusive peaks, cool blue for the most popular. The control bar at the bottom provides play/pause, a timeline scrubber, and a speed dial.

In [ ]:
function setup() {
  createCanvas(innerWidth, innerHeight);
  textFont('monospace');
  textAlign(CENTER, CENTER);

  baseY = height * 0.82;
  maxDisplayH = height * 0.65;

  noiseSeed(42);
  randomSeed(42);

  // generate mountain display data
  let margin = 50;
  let totalWidth = width - margin * 2;
  let peakWidth = totalWidth / byHeight.length;

  for (let i = 0; i < byHeight.length; i++) {
    let p = byHeight[i];
    let displayH = map(p.height, 7800, 9100, maxDisplayH * 0.22, maxDisplayH);
    let displayW = peakWidth - params.mountainGap;
    let displayX = margin + (i + 0.5) * peakWidth;

    noiseSeed(p.seed * 137);
    let profile = generateProfile(p.seed);
    let ascentIdx = byAscent.indexOf(p);

    mountainData.push({
      ...p,
      displayX,
      displayW,
      displayH,
      profile,
      ascentIdx,
    });
  }

  // animation timeline
  riseEnd = byAscent.length * params.framesPerPeak;
  holdEnd = riseEnd + params.holdFrames;
  totalFrames = holdEnd + params.transitionFrames;

  // control bar
  let bar = createDiv('');
  bar.position(0, height - 48);
  bar.style('width', '100%');
  bar.style('display', 'flex');
  bar.style('align-items', 'center');
  bar.style('gap', '10px');
  bar.style('padding', '10px 20px');
  bar.style('background', 'rgba(5, 6, 18, 0.92)');
  bar.style('backdrop-filter', 'blur(8px)');
  bar.style('box-sizing', 'border-box');
  bar.style('font-family', 'monospace');
  bar.style('color', '#b4bed2');
  bar.style('font-size', '13px');
  bar.style('border-top', '1px solid rgba(100,130,180,0.08)');

  playBtn = createButton('\u23F8');
  playBtn.parent(bar);
  playBtn.style('background', 'none');
  playBtn.style('border', '1px solid rgba(100,130,180,0.25)');
  playBtn.style('color', '#b4bed2');
  playBtn.style('font-size', '16px');
  playBtn.style('cursor', 'pointer');
  playBtn.style('padding', '2px 10px');
  playBtn.style('border-radius', '4px');
  playBtn.mousePressed(() => {
    if (currentIndex >= totalFrames) {
      currentIndex = 0;
      playing = true;
    } else {
      playing = !playing;
    }
    playBtn.html(playing ? '\u23F8' : '\u25B6');
  });

  timeline = createSlider(0, totalFrames, 0, 1);
  timeline.parent(bar);
  timeline.style('flex', '1');
  timeline.style('accent-color', '#e9c46a');
  timeline.input(() => {
    playing = false;
    playBtn.html('\u25B6');
    currentIndex = timeline.value();
  });

  let speedLabel = createSpan('Speed');
  speedLabel.parent(bar);
  speedSlider = createSlider(0.2, 1.4, 0.8, 0.1);
  speedSlider.parent(bar);
  speedSlider.style('width', '80px');
  speedSlider.style('accent-color', '#a8dadc');
}

function formatSummitCount(n, colWidth) {
  if (colWidth >= 90) return n.toLocaleString() + ' summits';
  if (n >= 1000) return (n / 1000).toFixed(1).replace(/\.0$/, '') + 'K';
  return String(n);
}

function draw() {
  if (width === 0 || !speedSlider) return;

  background(...params.bgColor);

  // advance
  if (playing) {
    currentIndex = min(currentIndex + speedSlider.value(), totalFrames);
    timeline.value(floor(currentIndex));
    if (currentIndex >= totalFrames) {
      playing = false;
      playBtn.html('\u25B6');
    }
  }

  // color blend (0 during rise/hold, 0→1 during transition)
  let colorBlend = 0;
  if (currentIndex > holdEnd) {
    colorBlend = easeInOutCubic(constrain((currentIndex - holdEnd) / params.transitionFrames, 0, 1));
  }

  // sky gradient — deep atmospheric twilight
  let ctx = drawingContext;
  ctx.save();
  let skyGrad = ctx.createLinearGradient(0, 0, 0, baseY);
  skyGrad.addColorStop(0, 'rgb(5, 6, 18)');
  skyGrad.addColorStop(0.25, 'rgb(10, 16, 38)');
  skyGrad.addColorStop(0.45, 'rgb(16, 26, 52)');
  skyGrad.addColorStop(0.60, 'rgb(22, 36, 65)');
  skyGrad.addColorStop(0.75, 'rgb(30, 48, 78)');
  skyGrad.addColorStop(0.87, 'rgb(40, 58, 86)');
  skyGrad.addColorStop(0.95, 'rgb(50, 65, 92)');
  skyGrad.addColorStop(1, 'rgb(58, 72, 98)');
  ctx.fillStyle = skyGrad;
  ctx.fillRect(0, 0, width, baseY);
  ctx.restore();

  // soft horizon glow
  ctx.save();
  let horizonGlow = ctx.createRadialGradient(
    width * 0.5, baseY, 0,
    width * 0.5, baseY, width * 0.6
  );
  horizonGlow.addColorStop(0, 'rgba(90, 120, 160, 0.18)');
  horizonGlow.addColorStop(0.4, 'rgba(70, 100, 140, 0.08)');
  horizonGlow.addColorStop(1, 'rgba(0, 0, 0, 0)');
  ctx.fillStyle = horizonGlow;
  ctx.fillRect(0, 0, width, baseY);
  ctx.restore();

  // subtle high-altitude haze bands
  ctx.save();
  let haze1Y = baseY * 0.62;
  let h1 = ctx.createLinearGradient(0, haze1Y - 18, 0, haze1Y + 18);
  h1.addColorStop(0, 'rgba(60, 85, 120, 0)');
  h1.addColorStop(0.5, 'rgba(60, 85, 120, 0.035)');
  h1.addColorStop(1, 'rgba(60, 85, 120, 0)');
  ctx.fillStyle = h1;
  ctx.fillRect(0, haze1Y - 18, width, 36);

  let haze2Y = baseY * 0.78;
  let h2 = ctx.createLinearGradient(0, haze2Y - 22, 0, haze2Y + 22);
  h2.addColorStop(0, 'rgba(65, 90, 125, 0)');
  h2.addColorStop(0.5, 'rgba(65, 90, 125, 0.045)');
  h2.addColorStop(1, 'rgba(65, 90, 125, 0)');
  ctx.fillStyle = h2;
  ctx.fillRect(0, haze2Y - 22, width, 44);
  ctx.restore();

  // ground
  noStroke();
  fill(5, 6, 14);
  rect(0, baseY, width, height - baseY);

  // draw mountains
  for (let md of mountainData) {
    let startFrame = md.ascentIdx * params.framesPerPeak;
    let riseProgress = constrain((currentIndex - startFrame) / params.framesPerPeak, 0, 1);
    if (riseProgress > 0) {
      drawMountain(md, riseProgress, colorBlend);
    }
  }

  // atmospheric haze at mountain bases
  ctx.save();
  let hazeGrad = ctx.createLinearGradient(0, baseY - 40, 0, baseY + 5);
  hazeGrad.addColorStop(0, 'rgba(20, 30, 50, 0)');
  hazeGrad.addColorStop(0.6, 'rgba(20, 30, 50, 0.15)');
  hazeGrad.addColorStop(1, 'rgba(12, 18, 35, 0.3)');
  ctx.fillStyle = hazeGrad;
  ctx.fillRect(0, baseY - 40, width, 45);
  ctx.restore();

  // peak labels
  textAlign(CENTER, TOP);
  for (let md of mountainData) {
    let startFrame = md.ascentIdx * params.framesPerPeak;
    let riseProgress = constrain((currentIndex - startFrame) / params.framesPerPeak, 0, 1);
    if (riseProgress <= 0) continue;

    let labelAlpha = riseProgress < 1
      ? constrain(map(riseProgress, 0.4, 1, 0, 180), 0, 180)
      : 180;

    // peak name
    noStroke();
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], labelAlpha * 0.8);
    textSize(9);
    text(md.name, md.displayX, baseY + 6);

    // height
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], labelAlpha * 0.4);
    textSize(8);
    text(md.height + ' m', md.displayX, baseY + 18);

    // summit count during color phase
    if (colorBlend > 0.3) {
      let sc = summitScale(md.summits).rgb();
      let a = constrain((colorBlend - 0.3) / 0.7, 0, 1) * 220;
      fill(sc[0], sc[1], sc[2], a);
      textSize(9);
      text(formatSummitCount(md.summits, md.displayW), md.displayX, baseY + 30);
    }
  }

  // current year display during rise phase
  if (currentIndex <= riseEnd + 10) {
    let peakNum = min(floor(currentIndex / params.framesPerPeak), byAscent.length - 1);
    let activePeak = byAscent[max(peakNum, 0)];
    let fadeFactor = currentIndex > riseEnd ? max(0, 1 - (currentIndex - riseEnd) / 30) : 1;

    if (fadeFactor > 0.01) {
      let yearY = height * 0.10;

      ctx.save();
      let yg = ctx.createRadialGradient(width / 2, yearY, 0, width / 2, yearY, 80);
      yg.addColorStop(0, 'rgba(160, 190, 230, ' + 0.06 * fadeFactor + ')');
      yg.addColorStop(1, 'rgba(160, 190, 230, 0)');
      ctx.fillStyle = yg;
      ctx.beginPath();
      ctx.arc(width / 2, yearY, 80, 0, Math.PI * 2);
      ctx.fill();
      ctx.restore();

      fill(255, 255, 255, 230 * fadeFactor);
      noStroke();
      textAlign(CENTER, CENTER);
      textSize(48);
      text(activePeak.firstAscent, width / 2, yearY);

      fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 160 * fadeFactor);
      textSize(14);
      text(activePeak.name + '  \u00b7  ' + activePeak.height + ' m', width / 2, yearY + 34);

      fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 100 * fadeFactor);
      textSize(12);
      text(activePeak.climbers, width / 2, yearY + 52);
    }
  }

  // title in color phase
  if (colorBlend > 0.1) {
    let titleAlpha = map(colorBlend, 0.1, 0.6, 0, 230, true);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], titleAlpha);
    noStroke();
    textAlign(CENTER, CENTER);
    textSize(22);
    text('The Summit Circle', width / 2, height * 0.05);
    textSize(12);
    fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], titleAlpha * 0.5);
    text('Successful ascents per eight-thousander  \u00b7  1950\u20132024', width / 2, height * 0.05 + 26);
  }

  // hover tooltip
  let hovered = getHoveredPeak();
  if (hovered) {
    drawTooltip(hovered);
  }

  // attribution
  textAlign(LEFT, TOP);
  fill(params.labelColor[0], params.labelColor[1], params.labelColor[2], 45);
  textSize(11);
  text('Himalayan Database  \u00b7  as of 2024', 16, 16);
  textAlign(CENTER, CENTER);
}

In [ ]:
%show 100% 600px